# Approach 2 · A5 Final Experiment
## 1. Setup

Open this notebook in the team repository and use **Python 3.13.17** with the project requirements and CUDA 12.8. With `DEBUG=False`, **Run All** creates a new run and performs the complete experiment. Start with the original DPD-Small images and CSV files and the official pretrained DINOv2 backbone. All experiment models start with fresh heads.

Use `DEBUG=True` for a one-epoch training-only smoke test. The full run writes an environment snapshot and a copy of the requirements file.

In [1]:
DEBUG = False

from datetime import datetime, timezone
from pathlib import Path
from importlib import metadata
import platform
import copy
import hashlib
import json
import sys

if sys.version_info[:3] != (3, 13, 17):
    raise RuntimeError(f"Select the Python 3.13.17 notebook kernel. Current kernel: {sys.executable}.")

import pandas as pd
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
import timm
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

here = Path.cwd()
ROOT = next((p for p in (here, *here.parents) if (p / "common" / "dataset.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook inside the team repository.")
sys.path.insert(0, str(ROOT))

from common import config as team
from common.dataset import PlantDataset, get_loaders, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, harmonic_mean, save_confusion, top_k_accuracy

BACKBONE = "vit_base_patch14_dinov2.lvd142m"
REFERENCE_COMMIT = "0fece2daff4917431bb2a5b16b65718ab100d05c"
DEVICE = team.DEVICE
NUM_CROPS, NUM_DISEASES = team.NUM_CROPS, team.NUM_DISEASES
team.NUM_WORKERS = 4
BATCH_SIZE, EPOCHS = (8, 1) if DEBUG else (16, 20)
HEAD_LR, BACKBONE_LR, WEIGHT_DECAY = 1e-4, 1e-5, 1e-4
SMOOTHING = 0.1
DECOR_WEIGHT, SUPCON_WEIGHT, TEMPERATURE = 0.05, 0.10, 0.10
BLUR_PROB, BLUR_SIGMA = 0.15, 0.7
LOGIT_SCALE = 16.0
HARD_NEGATIVES, PAIR_MARGIN, RANK_WEIGHT, TRANSFER_WEIGHT = 3, 1.0, 0.35, 0.25
RUN_NAME = "approach2_a5_full_final" + ("_debug" if DEBUG else "")
CONTROL, PAIRED = "A5_Control", "A5_PairedDiseaseAttention"
STAGED = "A5_Paired_Staged"
RECIPES = {CONTROL: False, PAIRED: True, STAGED: True}
EARLY_CONTROL_EPOCH = 2
HEAD_WARMUP_EPOCHS = 1
PATCH_GRID, DONOR_TEMPERATURE = 8, 0.1
PAIRED_WEIGHT, ALIGNMENT_WEIGHT = 0.3, 0.1
GAMMAS = [step / 2 for step in range(25)]
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RESULTS_DIR = team.RESULTS_DIR / RUN_NAME / RUN_ID
DEVELOPMENT_DIR = RESULTS_DIR / "development"
FINAL_DIR = RESULTS_DIR / "final"
OFFICIAL_DIR = RESULTS_DIR / "official"
CHECKPOINT_DIR = team.CHECKPOINT_DIR / RUN_NAME / RUN_ID
LOCK = FINAL_DIR / "selection_lock.json"

requirements_path = ROOT / "requirements.txt"
requirements_bytes = requirements_path.read_bytes()
requirements = requirements_bytes.decode("utf-8-sig")
for entry in requirements.splitlines():
    entry = entry.split(" #", 1)[0].strip()
    if not entry or entry.startswith(("#", "--index-url", "--extra-index-url")):
        continue
    requirement = Requirement(entry)
    if requirement.marker and not requirement.marker.evaluate():
        continue
    try:
        installed_version = metadata.version(requirement.name)
    except metadata.PackageNotFoundError as error:
        raise RuntimeError(f"Install the repository requirements in this kernel: {entry}.") from error
    if not requirement.specifier.contains(installed_version, prereleases=True):
        raise RuntimeError(f"requirements.txt specifies {entry}, but this kernel has {installed_version}.")

if not DEBUG and (DEVICE.type != "cuda" or torch.version.cuda != "12.8"
                  or not torch.cuda.is_available()):
    raise RuntimeError("The full run requires the project CUDA 12.8 PyTorch build and a visible GPU.")

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "executable": sys.executable,
    "requirements_sha256": hashlib.sha256(requirements_bytes).hexdigest(),
    "cuda_build": torch.version.cuda,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None,
    "packages": dict(sorted((canonicalize_name(dist.metadata["Name"]), dist.version)
                            for dist in metadata.distributions() if dist.metadata["Name"])),
}

if not team.TRAIN_CSV.is_file() or not team.IMAGE_DIR.is_dir():
    raise FileNotFoundError("Training CSV or image directory is missing.")
if (team.NUM_SEEN_PAIRS, team.NUM_UNSEEN_PAIRS, team.NUM_PAIRS) != (323, 10, 333):
    raise ValueError("Use the shared 323/333 evaluation API.")
print(f"PyTorch {torch.__version__} · {DEVICE} · debug={DEBUG} · run {RUN_ID}")

PyTorch 2.9.1+cu128 · cuda · debug=False · run 20261127T113731797820Z


## 2. Data

Use the original A5 pair holdout procedure on the official training CSV. Development removes ten complete pairs from fit while preserving the individual crop and disease concepts. Five pairs and seen validation images guide tuning. The other five pairs and a separate seen validation subset are used for checking. Training donors come only from the current fit batch. Full refits restore all 8,000 training images. The two official test CSVs first enter in Section 6.

In [2]:
def image_key(value):
    result = Path(str(value).strip().replace("\\", "/"))
    if not result.name or result.is_absolute() or ".." in result.parts or ":" in str(result):
        raise ValueError(f"Expected a relative image filename, got {value!r}.")
    return result.as_posix()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(".json.tmp")
    temp.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_save(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(".pt.tmp")
    torch.save(value, temp)
    temp.replace(path)


def atomic_csv(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(".csv.tmp")
    frame.to_csv(temp, index=False)
    temp.replace(path)


def write_run_environment():
    atomic_json(DEVELOPMENT_DIR / "environment.json", environment)
    DEVELOPMENT_DIR.mkdir(parents=True, exist_ok=True)
    (DEVELOPMENT_DIR / "requirements.txt").write_bytes(requirements_bytes)


def proxy_split(fit_frame, val_frame):
    """Hold out complete training pairs from the development heads only."""
    grouped = fit_frame.groupby("pair_row", sort=True)
    rows = grouped.agg(n=("image", "size"), crop=("crop", "first"), disease=("disease", "first"))
    crops = rows.crop.value_counts().to_dict()
    diseases = rows.disease.value_counts().to_dict()
    eligible = rows[rows.n.ge(5)].sample(frac=1, random_state=team.SEED + 1)
    held = []
    for pair_row, row in eligible.iterrows():
        if crops[row.crop] > 1 and diseases[row.disease] > 1:
            held.append(int(pair_row))
            crops[row.crop] -= 1
            diseases[row.disease] -= 1
        if len(held) == 10:
            break
    if len(held) < 10:
        raise ValueError("Ten train-derived holdouts with fit support were not available.")
    seen = val_frame.loc[~val_frame.pair_row.isin(held)].sample(frac=1, random_state=team.SEED + 2)
    cut = len(seen) // 2
    frames = {
        "tune_seen": seen.iloc[:cut],
        "tune_proxy": train_full.loc[train_full.pair_row.isin(held[:5])],
        "check_seen": seen.iloc[cut:],
        "check_proxy": train_full.loc[train_full.pair_row.isin(held[5:])],
    }
    if any(group.empty for group in frames.values()):
        raise ValueError("Proxy validation needs seen and held-out images.")
    fit_mask = torch.ones(len(train_pairs), dtype=torch.bool)
    fit_mask[held] = False
    return fit_mask, frames


train_full = load_split(team.TRAIN_CSV)
train_full["image"] = train_full.image.map(image_key)
if not train_full.image.is_unique:
    raise ValueError("Duplicate training images.")
train_pairs = get_pairs()
if train_pairs.shape != (team.NUM_SEEN_PAIRS, 2):
    raise ValueError("Unexpected training pair table.")
train_pairs_device = train_pairs.to(DEVICE)
_, image_transform = get_transforms("imagenet")

settings = {
    "reference_commit": REFERENCE_COMMIT, "backbone": BACKBONE, "seed": team.SEED,
    "image_size": team.IMAGE_SIZE, "resize_size": team.RESIZE_SIZE, "val_fraction": team.VAL_FRACTION,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "head_lr": HEAD_LR, "backbone_lr": BACKBONE_LR,
    "weight_decay": WEIGHT_DECAY, "optimizer": "AdamW", "schedule": "CosineAnnealingLR",
    "label_smoothing": SMOOTHING, "trainable_backbone": ["blocks.11", "norm"],
    "decor_weight": DECOR_WEIGHT, "supcon_weight": SUPCON_WEIGHT, "temperature": TEMPERATURE,
    "blur_probability": BLUR_PROB, "blur_sigma": BLUR_SIGMA,
    "scoring": "raw_additive_cosine", "logit_scale": LOGIT_SCALE,
    "ranking_weight": RANK_WEIGHT, "ranking_margin": PAIR_MARGIN, "hard_negatives": HARD_NEGATIVES,
    "transfer_ranking_weight": TRANSFER_WEIGHT,
    "pair_weighting": "inverse_sqrt_fit_pair_frequency",
    "train_csv_sha256": sha256_file(team.TRAIN_CSV),
    "recipes": RECIPES, "early_control_epoch": EARLY_CONTROL_EPOCH,
    "staged_heads_only_epochs": HEAD_WARMUP_EPOCHS, "paired_weight": PAIRED_WEIGHT, "alignment_weight": ALIGNMENT_WEIGHT,
    "donor_temperature": DONOR_TEMPERATURE, "patch_grid": PATCH_GRID, "gamma_grid": GAMMAS,
    "requirements_sha256": environment["requirements_sha256"],
    "environment_sha256": hashlib.sha256(json.dumps(environment, sort_keys=True).encode()).hexdigest(),
    "python": sys.version.split()[0],
    "torch": str(torch.__version__), "timm": timm.__version__, "debug": DEBUG,
}

## 3. Model

| Recipe | Training |
| --- | --- |
| A5 Control | Original A5 with one trainable DINOv2 block and its final norm |
| A5 Paired | Original A5 plus same-disease, different-crop patch supervision |
| A5 Paired Staged | Same paired objective. Train the heads with the last block and norm frozen for epoch 1, then unfreeze them at the same low backbone learning rate as A5 Paired |

All arms share the initial pretrained backbone, initial heads, training batch and augmentation. The first 11 blocks are frozen. Each image is scored with the sum of its crop and disease logits. Paired supervision is an auxiliary training loss on real fit images. At inference, each model processes one image.

The paired loss adapts the patch matching idea from [Learning Attention as Disentangler](https://arxiv.org/abs/2303.15111). The staged schedule adapts the heads-first strategy in [Fine-Tuning can Distort Pretrained Features](https://arxiv.org/abs/2202.10054).

In [3]:
@torch.no_grad()
def frozen_prefix(backbone, images):
    tokens = backbone.patch_embed(images)
    tokens = backbone._pos_embed(tokens)
    tokens = backbone.norm_pre(backbone.patch_drop(tokens))
    for block in backbone.blocks[:-1]:
        tokens = block(tokens)
    return tokens


def augment_blur(images, generator):
    selected = torch.rand(len(images), generator=generator) < BLUR_PROB
    if not selected.any():
        return images
    selected = selected.to(images.device)
    augmented = images.clone()
    augmented[selected] = TF.gaussian_blur(images[selected], kernel_size=[5, 5],
                                           sigma=[BLUR_SIGMA, BLUR_SIGMA])
    return augmented


def center_logits(logits):
    # Remove the per-image offset to which categorical predictions are invariant.
    return logits - logits.mean(dim=1, keepdim=True)


def decorrelation(crop, disease):
    def standardize(x):
        x = x - x.mean(0)
        return x / (x.square().mean(0) + 1e-4).sqrt()

    cross_correlation = standardize(crop).T @ standardize(disease) / len(crop)
    return cross_correlation.square().mean()


def cross_crop_supcon(embedding, crop, disease):
    z = F.normalize(embedding, dim=1)
    same_disease = disease[:, None].eq(disease[None, :])
    positives = same_disease & crop[:, None].ne(crop[None, :])
    negatives = ~same_disease
    similarity = z @ z.T / TEMPERATURE
    normalizer = similarity.masked_fill(~(positives | negatives), -1e9).logsumexp(dim=1)
    counts = positives.sum(dim=1)
    positive_mean = similarity.masked_fill(~positives, 0.0).sum(dim=1) / counts.clamp_min(1)
    usable = (counts > 0) & negatives.any(dim=1)
    per_anchor = (normalizer - positive_mean).masked_fill(~usable, 0.0)
    return per_anchor.sum() / usable.sum().clamp_min(1), usable.sum()


class CosineHead(nn.Module):
    def __init__(self, width, classes):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(classes, width))
        nn.init.normal_(self.weight, std=0.02)

    def forward(self, features):
        return LOGIT_SCALE * F.linear(F.normalize(features, dim=1), F.normalize(self.weight, dim=1))


class ConceptHeads(nn.Module):
    def __init__(self, width, prefix_tokens):
        super().__init__()
        self.prefix_tokens = prefix_tokens
        self.attention = nn.Sequential(nn.Linear(width, 128), nn.Tanh(), nn.Linear(128, 1, bias=False))
        nn.init.zeros_(self.attention[-1].weight)
        self.disease_projection = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 256))
        self.crop_head = CosineHead(width, NUM_CROPS)
        self.disease_head = CosineHead(256, NUM_DISEASES)

    def forward(self, tokens, return_patches=False):
        crop_feature = tokens[:, 0]
        patches = tokens[:, self.prefix_tokens:]
        attention = self.attention(patches).softmax(dim=1)
        disease_feature = self.disease_projection((patches * attention).sum(dim=1))
        outputs = (self.crop_head(crop_feature), self.disease_head(disease_feature), disease_feature)
        return (*outputs, patches) if return_patches else outputs


class PatchPairNet(nn.Module):
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0,
                                          img_size=team.IMAGE_SIZE)
        self.backbone.requires_grad_(False)
        self.heads = ConceptHeads(self.backbone.num_features, self.backbone.num_prefix_tokens)

    def forward(self, images):
        return self.heads(self.backbone.forward_features(images))


class FineTuneTail(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.block = copy.deepcopy(model.backbone.blocks[-1]).requires_grad_(True)
        self.norm = copy.deepcopy(model.backbone.norm).requires_grad_(True)
        self.heads = copy.deepcopy(model.heads)

    def forward(self, tokens, return_patches=False):
        return self.heads(self.norm(self.block(tokens)), return_patches=return_patches)


@torch.no_grad()
def verify_prefix(model, tail, images):
    for split, regular in zip(tail(frozen_prefix(model.backbone, images)), model(images)):
        torch.testing.assert_close(split, regular, rtol=1e-5, atol=1e-6)


def concept_pair_scores(crop_logits, disease_logits, pairs):
    return crop_logits[:, pairs[:, 0]] + disease_logits[:, pairs[:, 1]]


class PairRankingObjective(nn.Module):
    def __init__(self, fit_pair_rows, pairs):
        super().__init__()
        count = torch.bincount(fit_pair_rows, minlength=len(pairs))
        active = count.gt(0)
        if active.sum().item() < 2:
            raise ValueError("Pair ranking needs at least two fit pairs.")
        self.register_buffer("active", active)
        self.register_buffer("pairs", pairs)
        self.register_buffer("weights", count.clamp_min(1).float().pow(-0.5))
        self.hard_count = min(HARD_NEGATIVES, int(active.sum()) - 1)

    def margins(self, crop, disease, pair_rows):
        scores = concept_pair_scores(crop, disease, self.pairs)
        target = scores.gather(1, pair_rows[:, None])
        negatives = scores.masked_fill(~self.active[None, :], -torch.inf)
        negatives = negatives.scatter(1, pair_rows[:, None], -torch.inf)
        hardest = negatives.topk(self.hard_count, dim=1).values
        return F.softplus(PAIR_MARGIN + hardest - target).mean(dim=1)

    def forward(self, crop, disease, crop_labels, disease_labels, pair_rows):
        weights = self.weights[pair_rows]
        weights = weights / weights.sum()
        crop_ce = F.cross_entropy(crop, crop_labels, label_smoothing=SMOOTHING)
        disease_ce = F.cross_entropy(disease, disease_labels, reduction="none", label_smoothing=SMOOTHING)
        real_rank = torch.dot(weights, self.margins(crop, disease, pair_rows))

        # Attach disease evidence from same-disease images of other crops to the anchor crop.
        # The supervised target remains the anchor's real fit pair.
        donors = disease_labels[:, None].eq(disease_labels) & crop_labels[:, None].ne(crop_labels)
        donor_counts = donors.sum(dim=1)
        recombined = (donors.to(disease.dtype) @ disease) / donor_counts.clamp_min(1)[:, None]
        available = donor_counts > 0
        transfer_weights = self.weights[pair_rows] * available
        transfer_weights = transfer_weights / transfer_weights.sum().clamp_min(1e-8)
        transfer_rank = torch.dot(transfer_weights, self.margins(crop, recombined, pair_rows))

        total = crop_ce + torch.dot(weights, disease_ce) + RANK_WEIGHT * real_rank + TRANSFER_WEIGHT * transfer_rank
        return total, real_rank, transfer_rank, available.sum()

def paired_patch_loss(heads, patches, student, crops, diseases, grid_size):
    eligible = diseases[:, None].eq(diseases[None, :]) & crops[:, None].ne(crops[None, :])
    anchors, donors = eligible.nonzero(as_tuple=True)
    if not len(anchors):
        return student.new_zeros(()), 0
    # Average donor evidence per anchor, so crops with many eligible donors do not gain extra weight.
    width = patches.shape[2]
    spatial = patches.transpose(1, 2).reshape(len(patches), width, *grid_size)
    spatial = F.adaptive_avg_pool2d(spatial, tuple(min(PATCH_GRID, size) for size in grid_size))
    spatial = spatial.flatten(2).transpose(1, 2)
    normalized = F.normalize(spatial.detach(), dim=-1)
    similarities = normalized[anchors] @ normalized[donors].transpose(1, 2)
    matches = (similarities.amax(dim=2) / DONOR_TEMPERATURE).softmax(dim=1)
    pooled = (matches.unsqueeze(-1) * spatial[anchors]).sum(dim=1)
    shared = heads.disease_projection(pooled)
    ce = F.cross_entropy(heads.disease_head(shared), diseases[anchors], reduction="none",
                         label_smoothing=SMOOTHING)
    alignment = 1 - F.cosine_similarity(student[anchors], shared.detach(), dim=1)
    counts = eligible.sum(dim=1)
    weights = counts[anchors].float().reciprocal()
    total = (weights * (PAIRED_WEIGHT * ce + ALIGNMENT_WEIGHT * alignment)).sum() / weights.sum()
    return total, int(counts.count_nonzero())

## 4. Training

Choose each recipe's epoch and the overall winner by raw tune harmonic Top-1, followed by proxy Top-1, seen Top-1 and earlier epoch. The staged recipe is eligible from epoch 2 onward so the selected staged checkpoint has actually fine-tuned the last block. Use the tune split to lock gamma separately for each recipe and epoch going to official test. Record the epoch-2 A5 Control and A5 Paired comparison regardless of the winner. The matched staged-versus-joint comparison is reported in development at the staged recipe's selected epoch. Check results do not influence selection.

`raw_check_deltas.csv` reports matched comparisons, including raw proxy Top-1 and the supplied-crop diagnostic. `training_history.csv` records donor coverage and whether the tail was trainable.

In [4]:
def selection_key(row):
    return (row["harmonic_top1"], row["proxy_top1"], row["seen_top1"], -row["epoch"])


def snapshot_tail(tail):
    return {name: value.detach().cpu().clone() for name, value in tail.state_dict().items()}


def make_optimizer(tail):
    return torch.optim.AdamW([
        {"params": tail.heads.parameters(), "lr": HEAD_LR},
        {"params": [*tail.block.parameters(), *tail.norm.parameters()], "lr": BACKBONE_LR},
    ], weight_decay=WEIGHT_DECAY)


def train_tail(recipe, tail, optimizer, objective, tokens, crop, disease, pair_rows, grid_size):
    extra = RECIPES[recipe]
    outputs = tail(tokens, return_patches=extra)
    crop_logits, disease_logits, disease_features = outputs[:3]
    pair_loss, ranking, transfer, transfers = objective(crop_logits, disease_logits, crop, disease, pair_rows)
    decor = decorrelation(center_logits(crop_logits), center_logits(disease_logits))
    supcon, anchors = cross_crop_supcon(disease_features, crop, disease)
    loss = pair_loss + DECOR_WEIGHT * decor + SUPCON_WEIGHT * supcon
    aux, donor_count = loss.new_zeros(()), 0
    if extra:
        aux, donor_count = paired_patch_loss(tail.heads, outputs[3], disease_features, crop, disease, grid_size)
        loss = loss + aux
    if not torch.isfinite(loss):
        raise FloatingPointError(f"Non-finite training loss in {recipe}.")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return (loss.item(), ranking.item(), transfer.item(), int(anchors), int(transfers), aux.item(), donor_count)


@torch.no_grad()
def cached_group(backbone, frame):
    rng = torch.Generator().manual_seed(team.SEED)
    loader = DataLoader(PlantDataset(frame, image_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0, generator=rng)
    return [(frozen_prefix(backbone, images.to(DEVICE)).cpu(), target.cpu())
            for images, _, _, target in loader]


@torch.no_grad()
def score_group(tail, batches):
    scores, labels = [], []
    for tokens, targets in batches:
        crop, disease, _ = tail(tokens.to(DEVICE))
        scores.append(concept_pair_scores(crop, disease, train_pairs_device).cpu())
        labels.append(targets)
    return torch.cat(scores), torch.cat(labels)


def metrics(seen, proxy, fit_mask, gamma=0.0):
    seen_s, seen_y = seen
    proxy_s, proxy_y = proxy
    seen_s, proxy_s = (value - gamma * fit_mask for value in (seen_s, proxy_s))
    s, u = top_k_accuracy(seen_s, seen_y, 1), top_k_accuracy(proxy_s, proxy_y, 1)
    winner = proxy_s.argmax(1)
    held_winner = proxy_s.masked_fill(fit_mask[None, :], -torch.inf).argmax(1)
    blocked = held_winner.eq(proxy_y) & fit_mask[winner]
    eligible = train_pairs[proxy_y, 0, None].eq(train_pairs[None, :, 0])
    same_crop_top1 = top_k_accuracy(proxy_s.masked_fill(~eligible, -torch.inf), proxy_y, 1)
    return {"seen_top1": s, "proxy_top1": u, "harmonic_top1": harmonic_mean(s, u),
            "seen_top5": top_k_accuracy(seen_s, seen_y, 5),
            "proxy_top5": top_k_accuracy(proxy_s, proxy_y, 5),
            "proxy_blocked_by_fit_pairs_pct": round(100 * float(blocked.float().mean()), 2),
            "proxy_true_crop_top1": same_crop_top1}


def objective_for(frame):
    targets = torch.as_tensor(frame.pair_row.to_numpy(copy=True), dtype=torch.long)
    return PairRankingObjective(targets, train_pairs).to(DEVICE)


def make_training_model():
    model = PatchPairNet(pretrained=True).to(DEVICE).eval()
    if len(model.backbone.blocks) != 12:
        raise ValueError("Expected the 12-block DINOv2 ViT-B/14.")
    tail = FineTuneTail(model).to(DEVICE).eval()
    return model, tail


def tail_is_trainable(recipe, epoch):
    return recipe != STAGED or epoch > HEAD_WARMUP_EPOCHS


def train_epoch(model, trials, objective, loader, blur_rng, stage, epoch, epochs, fit_mask=None):
    for recipe, (tail, _) in trials.items():
        enabled = tail_is_trainable(recipe, epoch)
        tail.block.requires_grad_(enabled)
        tail.norm.requires_grad_(enabled)
    sums = {recipe: [0.0] * 7 for recipe in trials}
    examples = 0
    for step, (images, crops, diseases, pair_rows) in enumerate(loader, 1):
        images = augment_blur(images.to(DEVICE), blur_rng)
        crops, diseases, pair_rows = (tensor.to(DEVICE) for tensor in (crops, diseases, pair_rows))
        if fit_mask is not None:
            keep = fit_mask[pair_rows]
            if not keep.any():
                continue
            images, crops, diseases, pair_rows = images[keep], crops[keep], diseases[keep], pair_rows[keep]
        if epoch == 1 and examples == 0:
            verify_prefix(model, next(iter(trials.values()))[0], images[:1])
        tokens = frozen_prefix(model.backbone, images)
        for recipe, (tail, optimizer) in trials.items():
            values = train_tail(recipe, tail, optimizer, objective, tokens, crops, diseases, pair_rows,
                                model.backbone.patch_embed.grid_size)
            loss, rank, transfer, anchors, transfers, aux, donors = values
            add = (loss * len(images), rank * len(images), transfer * transfers, anchors,
                   transfers, aux * donors, donors)
            for i, value in enumerate(add):
                sums[recipe][i] += value
        examples += len(images)
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"A5 {stage} epoch {epoch}/{epochs} · batch {step}/{len(loader)}", flush=True)
    if examples == 0:
        raise ValueError("No fit images were processed.")
    return {recipe: {"recipe": recipe, "epoch": epoch, "images_trained": examples,
                     "tail_trainable": tail_is_trainable(recipe, epoch), "train_loss": loss / examples, "train_ranking_loss": rank / examples,
                     "train_transfer_ranking_loss": transfer / max(transfers, 1),
                     "transfer_anchor_fraction": transfers / examples, "supcon_anchor_fraction": anchors / examples,
                     "paired_loss": aux / max(donors, 1), "paired_anchor_fraction": donors / examples}
            for recipe, (loss, rank, transfer, anchors, transfers, aux, donors) in sums.items()}


def run_development():
    write_run_environment()
    team.set_seed()
    train, val, test_seen, test_unseen = get_loaders(BATCH_SIZE, norm="imagenet", debug=DEBUG)
    if test_seen is not None or test_unseen is not None:
        raise RuntimeError("Development must use only the training CSV.")
    train.pin_memory = False
    model, control = make_training_model()
    tails = {CONTROL: control, PAIRED: copy.deepcopy(control), STAGED: copy.deepcopy(control)}
    trials = {recipe: (tail, make_optimizer(tail)) for recipe, tail in tails.items()}
    schedules = [torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
                 for _, optimizer in trials.values()]
    fit_frame, fit_mask, fit_device = train.dataset.frame, None, None
    if not DEBUG:
        fit_mask, frames = proxy_split(fit_frame, val.dataset.frame)
        fit_device = fit_mask.to(DEVICE)
        fit_frame = fit_frame.loc[fit_frame.pair_row.isin(torch.where(fit_mask)[0].tolist())]
        tune_cache = [cached_group(model.backbone, frames[key]) for key in ("tune_seen", "tune_proxy")]
    objective = objective_for(fit_frame)
    if fit_mask is not None and not torch.equal(objective.active, fit_device):
        raise RuntimeError("Proxy holdout and fit pairs do not match.")

    blur_rng = torch.Generator().manual_seed(team.SEED + 172)
    best, history = {}, []
    early = None
    for epoch in range(1, EPOCHS + 1):
        rows = train_epoch(model, trials, objective, train, blur_rng, "development", epoch, EPOCHS, fit_device)
        for schedule in schedules:
            schedule.step()
        if not DEBUG:
            tune = {recipe: [score_group(tail, cache) for cache in tune_cache] for recipe, tail in tails.items()}
            for recipe, row in rows.items():
                row.update(metrics(*tune[recipe], fit_mask))
            improving = [recipe for recipe, row in rows.items()
                         if tail_is_trainable(recipe, epoch)
                         and (recipe not in best or selection_key(row) > selection_key(best[recipe]["rows"][recipe]))]
            if improving or epoch == EARLY_CONTROL_EPOCH:
                point = {"rows": rows, "states": {key: snapshot_tail(tail) for key, tail in tails.items()},
                         "scores": tune}
                for recipe in improving:
                    best[recipe] = point
                if epoch == EARLY_CONTROL_EPOCH:
                    early = point
        history.extend(rows.values())
        for row in rows.values():
            print(row, flush=True)

    if DEBUG:
        return finalize_development(history, best, early)
    del tune_cache
    return finalize_development(history, best, early, model, tails, frames, fit_frame, fit_mask)


def finalize_development(history, best, early, model=None, tails=None, frames=None, fit_frame=None, fit_mask=None):
    write_run_environment()
    atomic_csv(pd.DataFrame(history), DEVELOPMENT_DIR / 'training_history.csv')
    if DEBUG:
        return None
    if early is None or set(best) != set(RECIPES):
        raise RuntimeError("Development must include epoch 2 and at least one unfrozen staged epoch.")
    for recipe in (PAIRED, STAGED):
        if not any(row["recipe"] == recipe and row["paired_anchor_fraction"] > 0 for row in history):
            raise RuntimeError(f"No eligible paired images were processed by {recipe}.")
    winner = max(best, key=lambda name: selection_key(best[name]["rows"][name]))
    selected = best[winner]
    checks, grid, selections, deltas, matched_rows = [], [], {}, [], []

    # Decide all final model IDs and their recipe/epoch combinations before looking at check results.
    comparisons = {
        "epoch2_paired_supervision": (early, CONTROL, PAIRED),
        "winner_vs_control": (selected, CONTROL, winner),
    }
    final_points = {}
    for point, left, right in comparisons.values():
        for recipe in dict.fromkeys((left, right)):
            epoch = point["rows"][recipe]["epoch"]
            final_points[f"{recipe}_e{epoch:02d}"] = (recipe, point)

    def score_point(recipe, point):
        row = point["rows"][recipe]
        candidates = [{"recipe": recipe, "epoch": row["epoch"], "gamma": gamma,
                       **metrics(*point["scores"][recipe], fit_mask, gamma)} for gamma in GAMMAS]
        gamma = max(candidates, key=lambda row: (row["harmonic_top1"], row["proxy_top1"],
                                                row["seen_top1"], -row["gamma"]))["gamma"]
        grid.extend(candidates)
        return {"selection": row, "gamma": gamma}

    for recipe, point in best.items():
        selections[recipe] = score_point(recipe, point)
    final_models = {}
    for model_id, (recipe, point) in final_points.items():
        current = selections[recipe] if point is best[recipe] else score_point(recipe, point)
        final_models[model_id] = {"recipe": recipe, **current}
    winner_id = f"{winner}_e{selected['rows'][winner]['epoch']:02d}"
    pairs_to_compare = {
        name: [f"{recipe}_e{point['rows'][recipe]['epoch']:02d}" for recipe in (left, right)]
        for name, (point, left, right) in comparisons.items()
    }
    plan = {"recipe": winner, "winner_id": winner_id, "recipes": selections, "models": final_models,
            "official_comparisons": pairs_to_compare,
            "fit_rows": len(fit_frame), "train_csv_sha256": settings["train_csv_sha256"],
            "tune_proxy_pairs": sorted(frames["tune_proxy"].pair_row.unique().tolist()),
            "check_proxy_pairs": sorted(frames["check_proxy"].pair_row.unique().tolist())}
    atomic_json(DEVELOPMENT_DIR / "epoch_selection.json", plan)
    check_cache = [cached_group(model.backbone, frames[key]) for key in ("check_seen", "check_proxy")]

    def check_scores(recipe, point):
        tails[recipe].load_state_dict(point["states"][recipe])
        return [score_group(tails[recipe], cache) for cache in check_cache]

    for recipe, point in best.items():
        scores = check_scores(recipe, point)
        gamma = selections[recipe]["gamma"]
        for mode, penalty in (("raw", 0.0), ("calibrated", gamma)):
            checks.append({"recipe": recipe, "epoch": point["rows"][recipe]["epoch"],
                           "scoring": mode, "gamma": penalty, **metrics(*scores, fit_mask, penalty)})
    for label, point, left, right in (
        ("epoch2_paired_supervision", early, CONTROL, PAIRED),
        ("staged_vs_joint_at_staged_epoch", best[STAGED], PAIRED, STAGED),
        ("winner_epoch_paired_vs_control", selected, CONTROL, PAIRED),
    ):
        matched = {recipe: metrics(*check_scores(recipe, point), fit_mask) for recipe in (left, right)}
        epoch = point["rows"][right]["epoch"]
        for recipe, values in matched.items():
            matched_rows.append({"comparison": label, "recipe": recipe, "epoch": epoch, **values})
        deltas.append({"comparison": label, "recipe": right, "reference": left, "epoch": epoch,
                       **{f"{key}_delta_pp": round(matched[right][key] - matched[left][key], 2)
                          for key in matched[left]}})
    atomic_csv(pd.DataFrame(deltas), DEVELOPMENT_DIR / 'raw_check_deltas.csv')
    atomic_csv(pd.DataFrame(matched_rows), DEVELOPMENT_DIR / 'raw_matched_check.csv')
    atomic_csv(pd.DataFrame(checks), DEVELOPMENT_DIR / 'independent_proxy_check.csv')
    atomic_csv(pd.DataFrame(grid).drop_duplicates(), DEVELOPMENT_DIR / 'calibration_tune_grid.csv')
    table = pd.DataFrame([{**value["selection"], "locked_gamma": value["gamma"], "selected": recipe == winner}
                          for recipe, value in selections.items()])
    atomic_csv(table, DEVELOPMENT_DIR / 'selected_models.csv')
    display(table)
    return plan


development = run_development()

A5 development epoch 1/20 · batch 1/452
A5 development epoch 1/20 · batch 50/452
A5 development epoch 1/20 · batch 100/452
A5 development epoch 1/20 · batch 150/452
A5 development epoch 1/20 · batch 200/452
A5 development epoch 1/20 · batch 250/452
A5 development epoch 1/20 · batch 300/452
A5 development epoch 1/20 · batch 350/452
A5 development epoch 1/20 · batch 400/452
A5 development epoch 1/20 · batch 450/452
A5 development epoch 1/20 · batch 452/452
{'recipe': 'A5_Control', 'epoch': 1, 'images_trained': 6788, 'tail_trainable': True, 'train_loss': 6.0187301218123315, 'train_ranking_loss': 2.0420023201666795, 'train_transfer_ranking_loss': 0.9463194614474972, 'transfer_anchor_fraction': 0.22097819681791397, 'supcon_anchor_fraction': 0.22097819681791397, 'paired_loss': 0.0, 'paired_anchor_fraction': 0.0, 'seen_top1': 68.41, 'proxy_top1': 20.87, 'harmonic_top1': 31.98, 'seen_top5': 95.33, 'proxy_top5': 29.57, 'proxy_blocked_by_fit_pairs_pct': 71.3, 'proxy_true_crop_top1': 23.48}
{'rec

,recipe,epoch,images_trained,tail_trainable,train_loss,train_ranking_loss,train_transfer_ranking_loss,transfer_anchor_fraction,supcon_anchor_fraction,paired_loss,paired_anchor_fraction,seen_top1,proxy_top1,harmonic_top1,seen_top5,proxy_top5,proxy_blocked_by_fit_pairs_pct,proxy_true_crop_top1,locked_gamma,selected
0,A5_Control,10,6788,True,1.951435,0.118442,0.034803,0.213318,0.213318,0.000000,0.000000,86.81,23.48,36.96,99.45,53.91,66.09,28.70,5.5,False
1,A5_PairedDiseaseAttention,2,6788,True,4.034026,0.927672,0.266064,0.215380,0.215380,0.453098,0.215380,73.35,25.22,37.53,97.25,33.04,67.83,29.57,5.0,False
2,A5_Paired_Staged,13,6788,True,2.144168,0.093884,0.029847,0.213465,0.213465,0.303981,0.213465,87.91,24.35,38.14,99.18,48.70,66.09,28.70,5.5,True


## 5. Final evaluation preparation

Refit **A5 Control and A5 Paired through epoch 2**, plus the selected recipe and its epoch-matched A5 Control. A recipe runs once through its latest required epoch and saves all requested snapshots along the way. Duplicate recipe/epoch combinations share a checkpoint. Each refit starts from the original pretrained backbone and fresh heads and uses the entire training CSV. Lock the comparison plan, gamma values and checkpoint hashes before reading official test metadata.

In [5]:
def checkpoint_path(model_id):
    return CHECKPOINT_DIR / model_id / "model.pt"


def read_final_lock():
    lock = json.loads(LOCK.read_text(encoding="utf-8"))
    if (lock["settings"] != settings or lock["development"] != development
            or lock["seen_pairs"] != train_pairs.tolist()
            or set(lock["models"]) != set(development["models"])
            or sha256_file(team.TRAIN_CSV) != settings["train_csv_sha256"]):
        raise RuntimeError("Final training inputs or model inventory differ from the locked plan.")
    for model_id, record in lock["models"].items():
        if sha256_file(checkpoint_path(model_id)) != record["checkpoint_sha256"]:
            raise RuntimeError(f"Checkpoint changed after locking: {model_id}.")
    return lock


def export_model(model, tail, model_id, spec):
    state = {key: value.detach().cpu() for key, value in model.state_dict().items()}
    modules = ((f"backbone.blocks.{len(model.backbone.blocks) - 1}", tail.block),
               ("backbone.norm", tail.norm), ("heads", tail.heads))
    for prefix, module in modules:
        for key, value in module.state_dict().items():
            name = f"{prefix}.{key}"
            if name not in state or state[name].shape != value.shape:
                raise RuntimeError(f"Cannot merge tail tensor {name}.")
            state[name] = value.detach().cpu()
    recipe = spec["recipe"]
    epoch = spec["selection"]["epoch"]
    record = {"model_id": model_id, **spec, "paired_supervision": RECIPES[recipe],
              "staged": recipe == STAGED, "refit_epochs": epoch,
              "initialization": "original_pretrained_dinov2_fresh_heads", "images_per_epoch": len(train_full)}
    path = checkpoint_path(model_id)
    path.parent.mkdir(parents=True, exist_ok=True)
    atomic_save(path, {"state_dict": state, **record})
    return {**record, "checkpoint_sha256": sha256_file(path), "checkpoint_bytes": path.stat().st_size}


def refit_full_train(development):
    if LOCK.is_file():
        return read_final_lock()
    if json.loads((DEVELOPMENT_DIR / "epoch_selection.json").read_text(encoding="utf-8")) != development:
        raise RuntimeError("The development plan changed before refit.")
    if sha256_file(team.TRAIN_CSV) != settings["train_csv_sha256"]:
        raise RuntimeError("Training CSV changed after development.")

    team.set_seed()
    model, initial_tail = make_training_model()
    milestones = {}
    for model_id, spec in development["models"].items():
        epoch = spec["selection"]["epoch"]
        if not 1 <= epoch <= EPOCHS or not tail_is_trainable(spec["recipe"], epoch):
            raise ValueError(f"Invalid refit epoch for {model_id}.")
        milestones.setdefault(spec["recipe"], {})[epoch] = model_id
    tails = {recipe: copy.deepcopy(initial_tail) for recipe in milestones}
    del initial_tail
    trials = {recipe: (tail, make_optimizer(tail)) for recipe, tail in tails.items()}
    schedules = {recipe: torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
                 for recipe, (_, optimizer) in trials.items()}
    train_transform, _ = get_transforms("imagenet")
    loader = DataLoader(PlantDataset(train_full, train_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=team.NUM_WORKERS, drop_last=False,
                        generator=torch.Generator().manual_seed(team.SEED))
    objective = objective_for(train_full)
    if not objective.active.all():
        raise RuntimeError("Full refit must cover the 323 training pairs.")
    blur_rng = torch.Generator().manual_seed(team.SEED + 172)
    total = max(max(epochs) for epochs in milestones.values())
    history, records = [], {}
    for epoch in range(1, total + 1):
        active = {recipe: trial for recipe, trial in trials.items() if epoch <= max(milestones[recipe])}
        rows = train_epoch(model, active, objective, loader, blur_rng, "full refit", epoch, total)
        for recipe, row in rows.items():
            if row["images_trained"] != len(train_full):
                raise RuntimeError("A refit epoch did not process every training image.")
            schedules[recipe].step()
            history.append(row)
            print(row, flush=True)
            if epoch in milestones[recipe]:
                model_id = milestones[recipe][epoch]
                records[model_id] = export_model(model, tails[recipe], model_id, development["models"][model_id])

    if records.keys() != development["models"].keys():
        raise RuntimeError("At least one required checkpoint is missing.")
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    atomic_csv(pd.DataFrame(history), FINAL_DIR / 'refit_history.csv')
    lock = {"settings": settings, "development": development, "seen_pairs": train_pairs.tolist(),
            "models": records}
    atomic_json(LOCK, lock)
    return lock


run = None if DEBUG else refit_full_train(development)
if run is not None:
    print("Locked checkpoints:", ", ".join(run["models"]), flush=True)

A5 full refit epoch 1/13 · batch 1/500
A5 full refit epoch 1/13 · batch 50/500
A5 full refit epoch 1/13 · batch 100/500
A5 full refit epoch 1/13 · batch 150/500
A5 full refit epoch 1/13 · batch 200/500
A5 full refit epoch 1/13 · batch 250/500
A5 full refit epoch 1/13 · batch 300/500
A5 full refit epoch 1/13 · batch 350/500
A5 full refit epoch 1/13 · batch 400/500
A5 full refit epoch 1/13 · batch 450/500
A5 full refit epoch 1/13 · batch 500/500
{'recipe': 'A5_Control', 'epoch': 1, 'images_trained': 8000, 'tail_trainable': True, 'train_loss': 5.8288369917869565, 'train_ranking_loss': 1.950328109204769, 'train_transfer_ranking_loss': 0.8290864227615746, 'transfer_anchor_fraction': 0.24875, 'supcon_anchor_fraction': 0.24875, 'paired_loss': 0.0, 'paired_anchor_fraction': 0.0}
{'recipe': 'A5_PairedDiseaseAttention', 'epoch': 1, 'images_trained': 8000, 'tail_trainable': True, 'train_loss': 6.367258013248444, 'train_ranking_loss': 2.017393481373787, 'train_transfer_ranking_loss': 0.84790786601

## 6. Official test

After verifying every checkpoint, append the 10 official unseen crop–disease identities to the 323 training pairs. Read image filenames and generate all models' raw and calibrated predictions before reading per-image ground truth. The primary report uses 333-way raw predictions (`gamma=0`). Report seen, unseen and combined Top-1 and Top-5, harmonic mean, and seen confusion and robustness diagnostics. Restricted 323-way and 10-way results and calibrated predictions are supplied separately.

In [6]:
def official_vocabulary():
    # The final shortlist uses only concept identities and discards per-image row order.
    vocabulary = pd.read_csv(team.TEST_UNSEEN_CSV, header=None, usecols=[1, 2],
                             dtype=str, keep_default_na=False)
    vocabulary.columns = ["crop", "disease"]
    if not vocabulary.empty and not vocabulary.iloc[0]["crop"].strip().isdigit():
        vocabulary = vocabulary.iloc[1:]
    vocabulary = (vocabulary.astype("int64").drop_duplicates()
                  .sort_values(["crop", "disease"]))
    extra_pairs = torch.as_tensor(vocabulary.to_numpy(copy=True), dtype=torch.long)
    pairs = torch.cat((train_pairs, extra_pairs))
    if (len(extra_pairs) != team.NUM_UNSEEN_PAIRS or len(pairs.unique(dim=0)) != team.NUM_PAIRS
            or not set(extra_pairs[:, 0].tolist()) <= set(train_pairs[:, 0].tolist())
            or not set(extra_pairs[:, 1].tolist()) <= set(train_pairs[:, 1].tolist())):
        raise ValueError("Expected distinct official unseen pairs composed of training-derived concepts.")
    return pairs


def official_filenames(path):
    values = pd.read_csv(path, header=None, usecols=[0], dtype=str, keep_default_na=False).iloc[:, 0].tolist()
    header = {"image", "image_path", "image_name", "filename", "file_name", "file_path", "path"}
    if values and (values[0].strip().lower().replace(" ", "_") in header or
                   (len(values) > 1 and not Path(values[0]).suffix and Path(values[1]).suffix)):
        values = values[1:]
    names = [image_key(name) for name in values]
    if not names or len(set(names)) != len(names):
        raise ValueError(f"Expected unique image names in {path.name}.")
    return names


class OfficialImages(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        with Image.open(team.IMAGE_DIR / self.names[index]) as image:
            return image_transform(image.convert("RGB"))


def official_targets(csv_path, filenames, row_by_pair, *, seen):
    # This per-image projection is used after predictions. Ignore the official pair_label column.
    frame = pd.read_csv(csv_path, header=None, usecols=[0, 1, 2], dtype=str, keep_default_na=False)
    frame.columns = ["image", "crop", "disease"]
    if not frame.empty and not frame.iloc[0]["crop"].strip().isdigit():
        frame = frame.iloc[1:]
    if frame.image.map(image_key).tolist() != filenames:
        raise ValueError("Official targets and predictions have different image orders.")
    keys = frame[["crop", "disease"]].astype("int64").itertuples(index=False, name=None)
    try:
        labels = torch.tensor([row_by_pair[key] for key in keys], dtype=torch.long)
    except KeyError as error:
        raise ValueError(f"Official ground-truth pair absent from the frozen candidate table: {error}") from error
    if not ((labels < team.NUM_SEEN_PAIRS) == seen).all():
        raise ValueError("Official test split does not match seen membership.")
    return labels


def top_predictions(scores, offset=0):
    return {k: scores.topk(k, dim=1).indices + offset for k in (1, 5)}


def prediction_accuracy(predictions, labels):
    return round(100 * predictions.eq(labels[:, None]).any(1).float().mean().item(), 2)


@torch.inference_mode()
def predict_official(model, pairs, filenames, split, perturb=None):
    generator = torch.Generator().manual_seed(team.SEED)
    loader = DataLoader(OfficialImages(filenames), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, generator=generator)
    mean, std = (torch.tensor(values).view(1, 3, 1, 1) for values in team.NORM_STATS["imagenet"])
    pair_indices = pairs.to(DEVICE)
    scores, concepts = [], []
    for step, images in enumerate(loader, 1):
        if perturb is not None:
            images = (perturb(images * std + mean) - mean) / std
        crop, disease, _ = model(images.to(DEVICE))
        batch = concept_pair_scores(crop, disease, pair_indices).cpu()
        if batch.shape != (len(images), len(pairs)) or not torch.isfinite(batch).all():
            raise ValueError("Invalid raw pair scores.")
        scores.append(batch)
        concepts.append(torch.stack((crop.argmax(dim=1), disease.argmax(dim=1)), dim=1).cpu())
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"{split}: batch {step}/{len(loader)}", flush=True)
    return torch.cat(scores), torch.cat(concepts)


def predict_recipe(record, pairs, filenames):
    model_id = record["model_id"]
    saved = torch.load(checkpoint_path(model_id), map_location="cpu", weights_only=True)
    for key, value in record.items():
        if key not in ("checkpoint_sha256", "checkpoint_bytes") and saved[key] != value:
            raise RuntimeError(f"Checkpoint and selection lock differ for {model_id}: {key}.")
    model = PatchPairNet(pretrained=False)
    model.load_state_dict(saved["state_dict"], strict=True)
    model = model.to(DEVICE).eval()
    boundary = len(train_pairs)
    membership = torch.arange(len(pairs)) < boundary
    clean, heads = zip(*(predict_official(model, pairs, names, split)
                         for names, split in zip(filenames, ("seen", "unseen"))))
    perturbed = {}
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(team.SEED)
        for condition, transform in PERTURBATIONS.items():
            perturbed[condition], _ = predict_official(model, pairs, filenames[0], condition, transform)

    predictions = {}
    for rule, gamma in (("raw", 0.0), ("calibrated", record["gamma"])):
        seen_scores, unseen_scores = (scores - gamma * membership for scores in clean)
        predictions[rule] = {
            "gamma": gamma,
            "seen": top_predictions(seen_scores),
            "unseen": top_predictions(unseen_scores),
            "seen_only": top_predictions(seen_scores[:, :boundary]),
            "unseen_only": top_predictions(unseen_scores[:, boundary:], offset=boundary),
            "robust": {name: (scores - gamma * membership).topk(1, dim=1).indices
                       for name, scores in perturbed.items()},
        }

    return predictions, clean, heads


def score_trial(run, pairs, forward_pass, y_seen, y_unseen):
    predictions, clean, (seen_heads, unseen_heads) = forward_pass
    boundary = len(train_pairs)
    y_all = torch.cat((y_seen, y_unseen))

    reports, ranking, per_pair, robustness, restricted = [], [], [], [], []
    # Rows are unseen images and columns are candidate pairs.
    true_crop = pairs[y_unseen, 0]
    same_crop_candidates = true_crop[:, None].eq(pairs[None, :, 0])
    seen_head_hits = seen_heads.eq(pairs[y_seen])
    unseen_head_hits = unseen_heads.eq(pairs[y_unseen])
    head_top1 = {
        f"{split}_{head}_evidence_top1": round(100 * matches[:, col].float().mean().item(), 2)
        for split, matches in (("seen", seen_head_hits), ("unseen", unseen_head_hits))
        for col, head in enumerate(("crop", "disease"))
    }
    for name, prediction in predictions.items():
        gamma = prediction["gamma"]
        raw_scores = clean[1]
        crop_oracle = raw_scores.masked_fill(~same_crop_candidates, -torch.inf).topk(1, dim=1).indices
        raw_margins = raw_scores.gather(1, y_unseen[:, None]).squeeze(1) - raw_scores[:, :boundary].amax(dim=1)
        final_margins = raw_margins + gamma
        only_correct = prediction["unseen_only"][1].squeeze(1).eq(y_unseen)
        metrics = {"scoring": name, "gamma": gamma, "paired_supervision": run["paired_supervision"],
                   "staged": run["staged"], "selected_on_tune": run["model_id"] == development["winner_id"],
                   "epoch": run["selection"]["epoch"], "candidates": len(pairs), **head_top1,
                   "raw_true_crop_candidate_top1": prediction_accuracy(crop_oracle, y_unseen)}
        for k in (1, 5):
            metrics[f"seen_top{k}"] = prediction_accuracy(prediction["seen"][k], y_seen)
            metrics[f"unseen_top{k}"] = prediction_accuracy(prediction["unseen"][k], y_unseen)
            joined = torch.cat((prediction["seen"][k], prediction["unseen"][k]))
            metrics[f"overall_top{k}"] = prediction_accuracy(joined, y_all)
        metrics["harmonic_mean"] = harmonic_mean(metrics["seen_top1"], metrics["unseen_top1"])
        metrics["abs_seen_unseen_gap_pp"] = round(abs(metrics["seen_top1"] - metrics["unseen_top1"]), 2)
        seen_winner, unseen_winner = (prediction[split][1].squeeze(1) for split in ("seen", "unseen"))
        metrics["seen_predicted_unseen"] = int((seen_winner >= boundary).sum())
        for split, winner, target in (("seen", seen_winner, y_seen), ("unseen", unseen_winner, y_unseen)):
            concept_hits = pairs[winner].eq(pairs[target]).float().mean(dim=0) * 100
            metrics[f"{split}_test_crop_acc"], metrics[f"{split}_test_disease_acc"] = [
                round(float(value), 2) for value in concept_hits]
        reports.append(metrics)

        correct = unseen_winner.eq(y_unseen)
        predicts_seen = unseen_winner < boundary
        blocked = only_correct & predicts_seen
        same_crop = pairs[unseen_winner, 0].eq(pairs[y_unseen, 0])
        ranking.append({"scoring": name, "unseen_images": len(y_unseen), "joint_correct": int(correct.sum()),
                        "unseen_only_correct": int(only_correct.sum()), "unseen_predicted_seen": int(predicts_seen.sum()),
                        "blocked_by_seen": int(blocked.sum()), "blocked_same_crop": int((blocked & same_crop).sum()),
                        "blocked_other_crop": int((blocked & ~same_crop).sum()),
                        "median_true_pair_minus_top_seen_score": float(final_margins.median())})
        for index in range(boundary, len(pairs)):
            mask = y_unseen.eq(index)
            crop, disease = pairs[index].tolist()
            per_pair.append({"scoring": name, "crop": crop, "disease": disease, "support": int(mask.sum()),
                             "joint_correct": int((mask & correct).sum()),
                             "unseen_only_correct": int((mask & only_correct).sum()),
                             "blocked_by_seen": int((mask & blocked).sum())})
        for key, predictions_at_one in prediction["robust"].items():
            accuracy = prediction_accuracy(predictions_at_one, y_seen)
            robustness.append({"scoring": name, "condition": key, "seen_top1": accuracy,
                               "drop_pp": round(metrics["seen_top1"] - accuracy, 2)})
        for split, targets, count in (("seen_only", y_seen, boundary),
                ("unseen_only", y_unseen, len(pairs) - boundary)):
            restricted.append({"scoring": name, "split": split, "candidates": count,
                               **{f"top{k}": prediction_accuracy(prediction[split][k], targets) for k in (1, 5)}})
    return ({"results": reports, "restricted_candidates": restricted, "robustness": robustness,
             "unseen_ranking_diagnostics": ranking, "unseen_per_pair": per_pair},
            {name: pred["seen"][1].squeeze(1) for name, pred in predictions.items()})


def official_evaluation():
    if DEBUG or run is None or read_final_lock() != run:
        raise RuntimeError("Complete and lock all full-training refits before official evaluation.")
    pairs = official_vocabulary()
    paths = (team.TEST_SEEN_CSV, team.TEST_UNSEEN_CSV)
    filenames = tuple(official_filenames(path) for path in paths)
    seen_names, unseen_names = map(set, filenames)
    if seen_names & unseen_names or set(train_full.image) & (seen_names | unseen_names):
        raise ValueError("Training and official split filenames overlap.")

    # Every locked model generates both predictions before any per-image labels are read.
    predictions = {model_id: predict_recipe(record, pairs, filenames)
                   for model_id, record in run["models"].items()}
    rows = {tuple(pair): i for i, pair in enumerate(pairs.tolist())}
    y_seen = official_targets(paths[0], filenames[0], rows, seen=True)
    y_unseen = official_targets(paths[1], filenames[1], rows, seen=False)
    summary = {"selection": run, "selected_model_id": development["winner_id"],
               "planned_comparisons": development["official_comparisons"],
               "samples": {"seen": len(y_seen), "unseen": len(y_unseen)},
               "pair_order": "train_then_concept_ids", "candidate_pairs": pairs.tolist()}
    winners = {}
    for model_id, record in run["models"].items():
        report, predicted_seen = score_trial(record, pairs, predictions[model_id], y_seen, y_unseen)
        for key, values in report.items():
            summary.setdefault(key, []).extend({"model_id": model_id, "recipe": record["recipe"], **row}
                                               for row in values)
        winners.update({f"{model_id}_{rule}": values for rule, values in predicted_seen.items()})
    return summary, winners, y_seen


evaluation = None if DEBUG else official_evaluation()

seen: batch 1/125
seen: batch 50/125
seen: batch 100/125
seen: batch 125/125
unseen: batch 1/91
unseen: batch 50/91
unseen: batch 91/91
brightness: batch 1/125
brightness: batch 50/125
brightness: batch 100/125
brightness: batch 125/125
blur: batch 1/125
blur: batch 50/125
blur: batch 100/125
blur: batch 125/125
noise: batch 1/125
noise: batch 50/125
noise: batch 100/125
noise: batch 125/125
seen: batch 1/125
seen: batch 50/125
seen: batch 100/125
seen: batch 125/125
unseen: batch 1/91
unseen: batch 50/91
unseen: batch 91/91
brightness: batch 1/125
brightness: batch 50/125
brightness: batch 100/125
brightness: batch 125/125
blur: batch 1/125
blur: batch 50/125
blur: batch 100/125
blur: batch 125/125
noise: batch 1/125
noise: batch 50/125
noise: batch 100/125
noise: batch 125/125
seen: batch 1/125
seen: batch 50/125
seen: batch 100/125
seen: batch 125/125
unseen: batch 1/91
unseen: batch 50/91
unseen: batch 91/91
brightness: batch 1/125
brightness: batch 50/125
brightness: batch 100/125

## 7. Save

- `results/approach2_a5_full_final/<run_id>/development/` contains environment, training history, selection and raw check ablations.
- `results/approach2_a5_full_final/<run_id>/final/` contains full refit history and selection lock.
- `results/approach2_a5_full_final/<run_id>/official/` contains raw and calibrated tables and diagnostics. Read the raw rows of `matched_deltas.csv` for the epoch-2 and winner-versus-control comparisons.
- `checkpoints/approach2_a5_full_final/<run_id>/<model_id>/model.pt` contains a complete model checkpoint at each required recipe/epoch.

In [7]:
def save_results(summary, seen_winners, seen_targets):
    OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
    # Align the team confusion plot order only after predictions and metrics are fixed.
    pairs = [tuple(pair) for pair in summary["candidate_pairs"]]
    lookup = {tuple(pair): row for row, pair in enumerate(get_pairs(final_evaluation=True).tolist())}
    if set(pairs) != lookup.keys():
        raise ValueError("Candidate identities differ from the common evaluator.")
    to_common = torch.tensor([lookup[pair] for pair in pairs])
    canonical_targets = to_common[seen_targets]
    if (canonical_targets >= len(train_pairs)).any():
        raise ValueError("The common confusion plot expects seen target rows.")
    for name, winners in seen_winners.items():
        votes = F.one_hot(to_common[winners], num_classes=len(pairs)).float()
        stem = (OFFICIAL_DIR / name).relative_to(team.RESULTS_DIR).as_posix()
        OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
        save_confusion(votes, canonical_targets, stem, final_evaluation=True)

    table = pd.DataFrame(summary["results"])
    atomic_csv(table, OFFICIAL_DIR / 'official_comparison.csv')
    for mode, part in table.groupby("scoring", sort=False):
        atomic_csv(part, OFFICIAL_DIR / f'official_{mode}_results.csv')
    for name in ("restricted_candidates", "robustness", "unseen_ranking_diagnostics", "unseen_per_pair"):
        atomic_csv(pd.DataFrame(summary[name]), OFFICIAL_DIR / f'{name}.csv')
    for row in summary["results"]:
        atomic_json(OFFICIAL_DIR / f"{row['model_id']}_{row['scoring']}.json", row)
    keys = ("seen_top1", "unseen_top1", "overall_top1", "seen_top5", "unseen_top5", "overall_top5", "harmonic_mean")
    index = table.set_index(["model_id", "scoring"])
    deltas = []
    for label, (reference, trial) in summary["planned_comparisons"].items():
        if trial == reference:
            continue
        for mode in ("raw", "calibrated"):
            left, right = index.loc[(reference, mode)], index.loc[(trial, mode)]
            deltas.append({"comparison": label, "model_id": trial, "reference": reference, "scoring": mode,
                           **{f"{name}_delta_pp": round(float(right[name] - left[name]), 2) for name in keys}})
    atomic_csv(pd.DataFrame(deltas), OFFICIAL_DIR / 'matched_deltas.csv')
    summary["matched_deltas"] = deltas
    atomic_json(OFFICIAL_DIR / "official_report.json", summary)
    return table


if evaluation is not None:
    table = save_results(*evaluation)
    columns = ["model_id", "paired_supervision", "staged", "gamma", "epoch", "selected_on_tune",
               "seen_top1", "unseen_top1", "overall_top1", "seen_top5", "unseen_top5", "overall_top5", "harmonic_mean"]
    for mode, part in table.groupby("scoring", sort=False):
        print(f"{mode.capitalize()} results")
        display(part[columns])
    print("Selected checkpoint:", checkpoint_path(development["winner_id"]).resolve())
    print("All checkpoints:", CHECKPOINT_DIR.resolve())
    print("Development results:", DEVELOPMENT_DIR.resolve())
    print("Final selection and refit:", FINAL_DIR.resolve())
    print("Official results:", OFFICIAL_DIR.resolve())
else:
    print("Debug logs:", DEVELOPMENT_DIR.resolve())

Raw results


,model_id,paired_supervision,staged,gamma,epoch,selected_on_tune,seen_top1,unseen_top1,overall_top1,seen_top5,unseen_top5,overall_top5,harmonic_mean
0,A5_Control_e02,False,False,0.0,2,False,62.45,15.54,42.75,88.70,48.00,71.61,24.89
2,A5_PairedDiseaseAttention_e02,True,False,0.0,2,False,61.95,13.95,41.79,88.80,48.00,71.66,22.77
4,A5_Control_e13,False,False,0.0,13,False,72.70,10.01,46.37,92.70,44.06,72.27,17.60
6,A5_Paired_Staged_e13,True,True,0.0,13,True,71.80,9.74,45.74,93.15,45.51,73.14,17.15


Calibrated results


,model_id,paired_supervision,staged,gamma,epoch,selected_on_tune,seen_top1,unseen_top1,overall_top1,seen_top5,unseen_top5,overall_top5,harmonic_mean
1,A5_Control_e02,False,False,5.0,2,False,52.20,76.24,62.30,83.60,95.03,88.40,61.97
3,A5_PairedDiseaseAttention_e02,True,False,5.0,2,False,51.20,75.83,61.54,83.60,95.30,88.52,61.13
5,A5_Control_e13,False,False,5.5,13,False,64.35,73.27,68.10,90.25,96.75,92.98,68.52
7,A5_Paired_Staged_e13,True,True,5.5,13,True,63.20,73.83,67.66,90.30,96.96,93.10,68.10


Selected checkpoint: C:\Users\104394862\Documents\GitHub\COS30082-Plant-Disease-OOD\checkpoints\approach2_a5_full_final\20261127T113731797820Z\A5_Paired_Staged_e13\model.pt
All checkpoints: C:\Users\104394862\Documents\GitHub\COS30082-Plant-Disease-OOD\checkpoints\approach2_a5_full_final\20261127T113731797820Z
Development results: C:\Users\104394862\Documents\GitHub\COS30082-Plant-Disease-OOD\results\approach2_a5_full_final\20261127T113731797820Z\development
Final selection and refit: C:\Users\104394862\Documents\GitHub\COS30082-Plant-Disease-OOD\results\approach2_a5_full_final\20261127T113731797820Z\final
Official results: C:\Users\104394862\Documents\GitHub\COS30082-Plant-Disease-OOD\results\approach2_a5_full_final\20261127T113731797820Z\official
